# Three planes, which is where a reversed `kz` stops being a number

[`01_build.ipynb`](01_build.ipynb) checked the encoding arithmetically: every partition on the
lattice, `kz` held through each readout, TE constant to nanoseconds. All of that would also be
true of a volume reconstructed **inside out** along z, or with `ky` and `kz` exchanged.

So this notebook simulates the sequences `01` wrote, reconstructs them with a 3D FFT, and looks at
them. The value here is the picture; the assertions are deliberately few and weak, because a
rendered image is not a correctness gate — it is the layer that catches what the gates cannot.

**The four `.seq` files in `seq/` are two pairs, not four sequence variants**, and they answer
two different questions:

| notebook | question | pair |
|---|---|---|
| [`01_build.ipynb`](01_build.ipynb) | *which excitation mode?* | `gre_3d_nonselective.seq` · `gre_3d_slab.seq` |
| **this one** | *for the slab-selective mode, how is the z moment realised?* | `gre_3d_slab_combined.seq` · `gre_3d_slab_sequential.seq` |

Both files in the second pair are **slab-selective**. `gre_3d_slab_combined.seq` is the same
physics as `01`'s `gre_3d_slab.seq` — one combined z winder, the production path — and exists
separately only because a fair comparison needs both halves built at one common TR.

**Needs `seqcraft[sim]`.** Runs in about a minute.

In [ ]:
import os
import sys

SIM_THREADS = 4
for _var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'RAYON_NUM_THREADS',
             'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_var] = str(SIM_THREADS)

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

import seqcraft as sc

sys.path.insert(0, str(Path('..').resolve()))
import phantom as ph            # noqa: E402 -- the shared BrainWeb preparation, one directory up

torch.set_num_threads(SIM_THREADS)
plt.rcParams['figure.dpi'] = 72
warnings.filterwarnings('ignore', category=sc.SeqCraftWarning)

SEQ_DIR = Path('seq')
nominal = np.load(SEQ_DIR / 'gre_3d_nominal.npz')
NX, NY, NZ = (int(v) for v in nominal['matrix'])
FOV_MM = tuple(float(v) for v in nominal['fov_mm'])
TABLE = nominal['table']
print(f'{NX} x {NY} x {NZ} over {FOV_MM} mm, {len(TABLE)} repetitions')

In [ ]:
def simulate(path, obj, *, states=200):
    """Run one written .seq against `obj` and return the raw signal, one row per repetition."""
    seq = mr0.Sequence.import_file(str(path))
    graph = mr0.compute_graph(seq, obj, max_state_count=states, min_state_mag=1e-4)
    signal = mr0.execute_graph(graph, seq, obj, min_emitted_signal=1e-4, min_latent_signal=1e-5)
    return np.asarray(signal.detach().cpu().numpy()).reshape(len(TABLE), NX)


def reconstruct(signal):
    """Sort the repetitions into (kx, ky, kz) and take the 3D FFT.

    The table's indices are already zero-based with DC at ``matrix // 2``, which is exactly the
    order ``ifftshift`` expects -- so the centre convention the modules use is the same one the
    reconstruction assumes, and nothing is re-derived here.
    """
    cube = np.zeros((NX, NY, NZ), dtype=complex)
    for row, (line, partition) in enumerate(TABLE):
        cube[:, int(line), int(partition)] = signal[row]
    return np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(cube)))


obj = ph.built(matrix=NX, nz=NZ, n_coils=1)
print('phantom ready')

## The slab-selective volume, in three planes

`x` is left-right, `y` is anterior-posterior with the frontal lobes at high `y`, and `z` is the
slab axis — read off `../phantom.py`, which prepares the object once for every example.

In [ ]:
image = np.abs(reconstruct(simulate(SEQ_DIR / 'gre_3d_slab.seq', obj)))

def three_planes(volume, title):
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.6))
    views = (('axial  (x, y)', volume[:, :, NZ // 2].T),
             ('coronal (x, z)', volume[:, NY // 2, :].T),
             ('sagittal (y, z)', volume[NX // 2, :, :].T))
    for ax, (name, plane) in zip(axes, views):
        ax.imshow(plane, cmap='gray', origin='lower', aspect='auto')
        ax.set(title=name, xticks=[], yticks=[])
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig

three_planes(image, 'slab-selective 3D GRE, reconstructed with a 3D FFT')
print(f'volume {image.shape}, peak at {np.unravel_index(np.argmax(image), image.shape)}')

A head, the right way up, in a volume of the size the protocol asked for. That sentence is the
whole point of the notebook, and no k-space assertion in `01` makes it.

The two numeric checks worth keeping are weak on purpose:

In [ ]:
inside = ph.mask(matrix=NX, nz=NZ)
energy = float((image ** 2 * inside).sum() / (image ** 2).sum())

print(f'reconstructed shape        {image.shape}  (expected {(NX, NY, NZ)})')
print(f'energy inside the phantom  {energy:.3f}')
assert image.shape == (NX, NY, NZ)
assert energy > 0.9, 'the object is not where the encoding says it should be'

## Non-selective, for comparison

The same kernel with `slab_thickness_mm=None`. At this geometry the two reconstruct the same
volume, and that is worth stating carefully rather than dressing up: **the phantom is thinner
than both the excited slab and the encoded FOV**, so there is nothing outside the slab for the
selective pulse to leave out. The difference between the two paths here is in the z moment each
one has to realise, not in what the object looks like.

In [ ]:
plain = np.abs(reconstruct(simulate(SEQ_DIR / 'gre_3d_nonselective.seq', obj)))
three_planes(plain, 'non-selective 3D GRE -- same object, same geometry')

difference = np.abs(plain - image).max() / image.max()
print(f'largest difference between the two volumes: {difference:.3f} of the peak')
print('(identical here: see the note above -- and MRzero models the pulse as a rotation,')
print(' so the slab profile itself is not simulated either)')

## Sequential z against combined z

A slab-selective 3D acquisition needs two things from the z axis in the same window: the slab
rephaser that undoes the excitation, and the partition encode. They can be played one after the
other, or added and played as **one** gradient:

```text
A_z(p) = A_slab + A_partition(p)
```

`GRE3DTR` does the second, which is shorter. Below, the sequential arrangement is built by hand
from the same leaves, so the only thing that differs is **how** the required z moment is
realised.

In [ ]:
opts = pp.Opts(max_grad=20, grad_unit='mT/m', max_slew=120, slew_unit='T/m/s', B0=3.0,
               rf_dead_time=100e-6, rf_ringdown_time=30e-6, adc_dead_time=10e-6)
raster = sc.Raster(opts.grad_raster_time)

# **One TR for both.**  The sequential arrangement is inherently longer -- that is the point --
# so it cannot be stacked at the combined kernel's own minimum, and a first attempt that tried
# overlapped every repetition by exactly the 100 us being measured.  Both scans are therefore
# built here at one explicit TR that each can meet, so the only difference left is the z
# realisation and the echo time it buys.
COMMON_TR_S = 10e-3

tr = sc.modules.GRE3DTR(opts=opts, fov_mm=FOV_MM, matrix=(NX, NY, NZ),
                        slab_thickness_mm=float(nominal['slab_mm']),
                        flip_deg=float(nominal['flip_deg']),
                        bandwidth_hz_px=float(nominal['bandwidth_hz_px']),
                        tr_s=COMMON_TR_S)


def sequential_tr(line, partition, *, phase_deg=0.0, acquire=True):
    """The reference arrangement: the excitation rephases itself, then the partition encodes."""
    out = sc.LogicBlock('sequential').add(0.0, tr.exc(phase_deg=phase_deg))   # rephase=True
    start = float(raster.ceil(tr.exc().duration))
    out.add(start, tr.pe(line=line)).add(start, tr.pe_z(line=partition))
    out.add(start, tr.ro(acquire=acquire, phase_deg=phase_deg))
    # Not ``+ winder_s``: the readout block starts with its prephaser, designed at that duration,
    # so ``ro().duration`` already carries it.  The kernel had this same double-count once.
    tail = start + tr.ro().duration
    out.add(tail, tr.pe(line=line, rewind=True)).add(tail, tr.pe_z(line=partition, rewind=True))
    for block in tr.spoilers.values():                      # the same spoiler the kernel plays
        out.add(tail, block)
    tail_s = max(tr.pe(line=0, rewind=True).duration,
                 *(block.duration for block in tr.spoilers.values()))
    fill = float(raster.ceil(tr.tr_s - tail - tail_s))
    if fill < -1e-12:
        raise ValueError(f'the sequential TR needs {(tail + tail_s) * 1e3:.3f} ms, '
                         f'more than the {tr.tr_s * 1e3:.3f} ms TR')
    if fill > 1e-9:
        out.add(tail + tail_s, pp.make_delay(fill))
    return out


probe = sequential_tr(3, 6)
k_sequential = sc.kspace(probe, opts)
k_combined = sc.kspace(tr(line=3, partition=6), opts)
echo = tr.ro.echo_sample(0)
te_sequential = float(k_sequential['t_adc'][echo] - k_sequential['t_excitation'][0])
te_combined = float(k_combined['t_adc'][echo] - k_combined['t_excitation'][0])

print(f'sequential  kz at the echo {k_sequential["k_adc"][2, echo]:8.4f} 1/m   '
      f'TE {te_sequential * 1e3:.4f} ms   block {probe.duration * 1e3:.3f} ms')
print(f'combined    kz at the echo {k_combined["k_adc"][2, echo]:8.4f} 1/m   '
      f'TE {te_combined * 1e3:.4f} ms   block {tr(line=3, partition=6).duration * 1e3:.3f} ms')
print(f'\nthe same acquisition, {(te_sequential - te_combined) * 1e6:.0f} us of TE cheaper '
      f'per repetition')

In [ ]:
def phase_deg(n, increment=117.0):
    return 0.5 * increment * n * (n + 1)


def scan(make_tr):
    dummies = int(nominal['dummies'])
    out = sc.LogicBlock('scan')
    for n in range(dummies):
        out.add(n * tr.tr_s, make_tr(int(TABLE[0][0]), int(TABLE[0][1]),
                                     phase_deg=phase_deg(n), acquire=False))
    for index, (line, partition) in enumerate(TABLE):
        n = dummies + index
        out.add(n * tr.tr_s, make_tr(int(line), int(partition), phase_deg=phase_deg(n)))
    return out


# Both are slab-selective, and both are written here rather than reused from 01, because a fair
# comparison needs them at one common TR.  `gre_3d_slab_combined.seq` is the same physics as
# `01`'s `gre_3d_slab.seq` -- it exists only as the matched half of this pair.
for name, make_tr in (('slab_combined', lambda *a, **k: tr(line=a[0], partition=a[1], **k)),
                      ('slab_sequential', sequential_tr)):
    sc.compile(scan(make_tr), opts, name=f'gre_3d_{name}').write(
        str(SEQ_DIR / f'gre_3d_{name}.seq'))

combined_image = np.abs(reconstruct(simulate(SEQ_DIR / 'gre_3d_slab_combined.seq', obj)))
sequential_image = np.abs(reconstruct(simulate(SEQ_DIR / 'gre_3d_slab_sequential.seq', obj)))

three_planes(sequential_image, 'sequential z: slab rephaser, then partition encode')
scale = combined_image.max()
print(f'largest difference between the two volumes: '
      f'{np.abs(sequential_image - combined_image).max() / scale:.4f} of the peak')
print(f'mean absolute difference:                   '
      f'{np.abs(sequential_image - combined_image).mean() / scale:.4f}')

**0.0014 of the peak, and 0.0001 on average.**

Not zero, and it should not be: the combined realisation reaches the echo 100 µs earlier, so a
small T2\*-dependent amplitude difference is **physically expected**. The claim being made is not
that the two produce identical signal. It is that they perform the same acquisition:

| the same | may differ |
|---|---|
| intended k-space encoding | signal amplitude, because TE differs |
| geometry and orientation | |
| semantic centre and partition spacing | |

In other words the two arrangements differ in **how** the required z moment is realised, not in
**what** acquisition is performed.

## Summary

| | |
|---|---|
| the volume reconstructs as a head, in three planes, the right way up | which no k-space assertion in `01` demonstrates |
| the module's centre conventions are the reconstruction's | `ifftshift` on the table's own indices, with nothing re-derived |
| the combined-z realisation reaches the same image as the sequential one | at a shorter echo time per repetition |
| the slab is not visible at this geometry | because the phantom is thinner than both the slab and the encoded FOV — said plainly rather than staged |

The evidence here is a human looking at a picture, which is the right instrument for orientation
and the wrong one for anything numerical — the assertions that have to hold live in
`tests/modules/test_gre_3d_tr.py` and do not depend on this notebook running.